# EXP006_GPU_V1 — complete GPU-native GCN tuning

**One-time setup:** choose a GPU in **Runtime → Change runtime type**, then **Run all**.
Authorize your Google Drive when asked and select **EXP006_GPU_V1_input_bundle.zip**
when the upload prompt appears. The supplied bundle already contains the audited
source and all 81 TRAIN arrays. No Mac transfer script or historical results are required.

After setup, all stages run automatically: preflight → screening → new shortlist →
chronological robustness → GPU reference controls → confirmation → promotion → export.
**1,209 research fits. No NASA TEST or anomaly-label evaluation.**

Drive destination: `My Drive/NASA_EXP006_GPU_V1/` (separate from your original project).
Do not run two copies of this notebook. After a runtime termination, reconnect to
the **same GPU model** and Run all again. Completed fits are verified and skipped;
the active uncommitted fit restarts from its seed. A crashed session's lease expires
in ten minutes. A different GPU model/software/driver stops rather than mixing runs.

The first setup pins PyTorch 2.11.0+cu130, NumPy 2.1.3 and pandas 2.2.3 in an isolated
environment, locks all resolved packages and records the exact execution environment.
No CPU/CUDA equivalence claim is made. CPU EXP006 remains historical only.

**Colab can end a runtime before the study finishes.** This notebook preserves progress;
it cannot guarantee uninterrupted execution or reconnect itself. No keep-alive hacks.
Automatic engineering checks must pass before research training begins.


In [ ]:
# GPU + durable storage + immutable input (only initial Drive/upload interaction).
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path, PurePosixPath
from google.colab import drive, files

EXPECTED_98c47256dfa532435f7f2c6a2c873513305de0880b3845b1b8854f0fba74f21a256 = "98c47256dfa532435f7f2c6a2c873513305de0880b3845b1b8854f0fba74f21a"
BUNDLE_NAME = "EXP006_GPU_V1_input_bundle.zip"
if sys.version_info[:2] not in ((3, 12), (3, 13)):
    raise RuntimeError("This pinned environment supports Python 3.12/3.13. Do not change dependency versions silently.")
subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv"], check=True)
drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/NASA_EXP006_GPU_V1")
BASE.mkdir(parents=True, exist_ok=True)
DURABLE_BUNDLE = BASE / BUNDLE_NAME

def file_sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""): h.update(block)
    return h.hexdigest()

if not DURABLE_BUNDLE.exists():
    print("Select the supplied " + BUNDLE_NAME)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Supply exactly the one input bundle.")
    name, data = next(iter(uploaded.items()))
    if hashlib.sha256(data).hexdigest() != EXPECTED_98c47256dfa532435f7f2c6a2c873513305de0880b3845b1b8854f0fba74f21a256:
        raise RuntimeError("Input bundle checksum mismatch; no training started.")
    temporary = DURABLE_BUNDLE.with_suffix(".zip.uploading")
    with open(temporary, "wb") as f:
        f.write(data); f.flush(); os.fsync(f.fileno())
    if file_sha(temporary) != EXPECTED_98c47256dfa532435f7f2c6a2c873513305de0880b3845b1b8854f0fba74f21a256:
        raise RuntimeError("Drive copy verification failed")
    os.replace(temporary, DURABLE_BUNDLE)
    del uploaded, data
if file_sha(DURABLE_BUNDLE) != EXPECTED_98c47256dfa532435f7f2c6a2c873513305de0880b3845b1b8854f0fba74f21a256:
    raise RuntimeError("Saved input changed; refusing to continue.")

WORK = Path("/content/exp006_gpu_v1_work")
ROOT = WORK / "package"
WORK.mkdir(parents=True, exist_ok=True)
local_zip = WORK / BUNDLE_NAME
shutil.copy2(DURABLE_BUNDLE, local_zip)
if file_sha(local_zip) != EXPECTED_98c47256dfa532435f7f2c6a2c873513305de0880b3845b1b8854f0fba74f21a256: raise RuntimeError("Local bundle copy failed")
if ROOT.exists(): shutil.rmtree(ROOT)  # Reconstruct only disposable immutable input, never results.
ROOT.mkdir()
with zipfile.ZipFile(local_zip) as z:
    names = z.namelist()
    if len(names) != len(set(names)): raise RuntimeError("Duplicate archive entries")
    manifest = json.loads(z.read("PACKAGE_MANIFEST.json"))
    if set(names) != set(manifest["files"]) | {"PACKAGE_MANIFEST.json"}:
        raise RuntimeError("Unexpected archive members")
    for info in z.infolist():
        p = PurePosixPath(info.filename)
        if p.is_absolute() or ".." in p.parts or "\\" in info.filename:
            raise RuntimeError("Unsafe archive path")
        if (info.external_attr >> 16) & 0o170000 == 0o120000:
            raise RuntimeError("Symlinks are not accepted")
        payload = z.read(info)
        if info.filename != "PACKAGE_MANIFEST.json" and hashlib.sha256(payload).hexdigest() != manifest["files"][info.filename]:
            raise RuntimeError("File checksum mismatch: " + info.filename)
        target = ROOT.joinpath(*p.parts); target.parent.mkdir(parents=True, exist_ok=True); target.write_bytes(payload)
OUTPUT = BASE / "results"
OUTPUT.mkdir(exist_ok=True)
print("Verified immutable TRAIN-only input. Durable results:", OUTPUT)


In [ ]:
!python -m pip install -q virtualenv
!rm -rf /content/exp006_gpu_v1_work/venv
!python -m virtualenv /content/exp006_gpu_v1_work/venv

In [ ]:
# Isolated pinned environment; no kernel restart required.
import venv
ENV = WORK / "venv"
PYTHON = ENV / "bin/python"
if not PYTHON.exists(): venv.EnvBuilder(with_pip=True).create(ENV)
LOCK = BASE / "requirements-resolved.txt"
pip = [str(PYTHON), "-m", "pip"]
if LOCK.exists():
    subprocess.run(pip + ["install", "--disable-pip-version-check", "-r", str(LOCK),
                         "--extra-index-url", "https://download.pytorch.org/whl/cu130"], check=True)
else:
    subprocess.run(pip + ["install", "--disable-pip-version-check", "torch==2.11.0+cu130",
                         "--index-url", "https://download.pytorch.org/whl/cu130"], check=True)
    subprocess.run(pip + ["install", "--disable-pip-version-check", "numpy==2.1.3", "pandas==2.2.3"], check=True)
subprocess.run(pip + ["check"], check=True)
resolved = subprocess.check_output(pip + ["freeze", "--all"], text=True)
if LOCK.exists():
    if sorted(LOCK.read_text().splitlines()) != sorted(resolved.splitlines()):
        raise RuntimeError("Resolved packages differ from the frozen environment; do not proceed.")
else:
    temp = LOCK.with_suffix(".tmp"); temp.write_text(resolved); os.replace(temp, LOCK)
shutil.copy2(LOCK, OUTPUT / LOCK.name)
RUN_ENV = os.environ.copy()
RUN_ENV.update(PYTHONHASHSEED="17", CUBLAS_WORKSPACE_CONFIG=":4096:8",
               OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1",
               PYTHONUNBUFFERED="1", PYTHONDONTWRITEBYTECODE="1")
print("Environment pinned. Full dependency lock saved.")


In [ ]:
# Complete workflow; no manual shortlist choice or stage switching.
LOGS = OUTPUT / "logs"; LOGS.mkdir(exist_ok=True)
log_path = LOGS / (time.strftime("session_%Y%m%dT%H%M%S") + ".log")
command = [str(PYTHON), "-u", str(ROOT / "src/gpu_run.py"), "--output", str(OUTPUT)]
with open(log_path, "a", buffering=1) as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, env=RUN_ENV)
    try:
        for line in process.stdout:
            print(line, end=""); log.write(line); log.flush()
        returncode = process.wait()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=30)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
if returncode != 0:
    raise RuntimeError("Stopped safely. Completed fits remain on Drive. See " + str(log_path))
status = json.loads((OUTPUT / "STATUS.json").read_text())
if status.get("status") != "COMPLETE" or status.get("fits") != 1209:
    raise RuntimeError("Workflow did not pass final completeness checks.")
print((OUTPUT / "SUMMARY.md").read_text())


In [ ]:
# Small report download; full checkpoint export is already on Drive.
from IPython.display import display, Markdown
display(Markdown((OUTPUT / "SUMMARY.md").read_text()))
exports = OUTPUT / "exports"
for p in sorted(exports.glob("*.zip")):
    print(f"{p.name}: {p.stat().st_size / 1024**2:.1f} MiB — {p}")
print("Full results and checkpoints remain saved even if a browser download is blocked.")
try:
    files.download(str(exports / "EXP006_GPU_V1_summary.zip"))
except Exception as exc:
    print("Browser download unavailable; use the saved Drive export:", exc)


### Interpretation

Promotion is a TRAIN-only forecasting decision against **GPU refits of EXP002's
architecture**, preserving the ≥2% mission-mean, all-three-seed direction, and
earlier-origin robustness gates. It says nothing yet about anomaly detection.
If neither candidate passes for a mission, retain EXP002's architecture and stop
the replacement search. Historical experiments and NASA anomaly results remain intact.

The full versioned protocol, original frozen sources, trainer diff and engineering
tests are included in the input bundle and exported reproducibility folder.
